# 01 — Exploratory Data Analysis

Goals:
- Understand the dataset shape, column types, and missingness
- Inspect the class distribution (delayed vs on-time)
- Identify temporal patterns (delay rate by hour, day, month)
- Check for obvious data quality issues

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_raw, BLACKLISTED_COLUMNS

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
print('Libraries loaded.')

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
DATA_PATH = '../data/flights_2023.csv'  # adjust path as needed
df = load_raw(DATA_PATH)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# ── Basic info ─────────────────────────────────────────────────────────────
print('Dtypes:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# ── Class distribution ─────────────────────────────────────────────────────
vc = df['y'].value_counts()
print(f"On-time (0): {vc[0]:,}  ({100*vc[0]/len(df):.1f}%)")
print(f"Delayed (1): {vc[1]:,}  ({100*vc[1]/len(df):.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['On-Time', 'Delayed'], [vc[0], vc[1]], color=['#22c55e', '#ef4444'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── Delay rate by month ─────────────────────────────────────────────────────
monthly = df.groupby('Month')['y'].agg(['mean', 'count']).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(monthly['Month'], monthly['mean'], color='#3b82f6')
axes[0].set_title('Delay Rate by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Delay Rate')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

axes[1].bar(monthly['Month'], monthly['count'], color='#8b5cf6')
axes[1].set_title('Flight Count by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Count')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ── Delay rate by departure hour ──────────────────────────────────────────
df['dep_hour'] = (df['CRS_DEP_TIME'].astype(int) // 100).clip(0, 23)
hourly = df.groupby('dep_hour')['y'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hourly.index, hourly.values, color='#f59e0b')
ax.set_title('Delay Rate by Departure Hour')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Delay Rate')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [ ]:
# ── Delay rate by airline ──────────────────────────────────────────────────
airline_stats = (
    df.groupby('OP_CARRIER')['y']
    .agg(['mean', 'count'])
    .query('count >= 1000')
    .sort_values('mean', ascending=False)
    .reset_index()
)
print(airline_stats[['OP_CARRIER', 'mean', 'count']].to_string(index=False))

In [ ]:
# ── ArrDelay distribution ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(df['ArrDelay'].clip(-60, 180), bins=100, color='#6366f1', edgecolor='white', linewidth=0.3)
ax.axvline(15, color='red', linestyle='--', label='15-min threshold')
ax.axvline(0,  color='gray', linestyle='--', alpha=0.5, label='On-time')
ax.set_title('Arrival Delay Distribution (clipped at -60 to 180 min)')
ax.set_xlabel('Arrival Delay (minutes)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()